# Module 03 — Classical image processing

> **Aquatic Guardian · Learn computer vision** — Module 03 of 09.  
> **Time:** about 2.5 hours. **Runs on:** free Google Colab, CPU is enough. **Skip if:** you can implement convolution, Otsu thresholding and connected components, and measure a blob's length with PCA.

**How to use this notebook.** In Colab choose **File → Save a copy in Drive** first, so your work is kept.
Then **Runtime → Run all**. Every cell runs without errors before you touch anything; the exercises print
feedback rather than crashing. Cells are marked:

- ▶ **Run and read** — nothing to change.
- ✏️ **Predict, then change** — write down what you expect, change one value, re-run.
- 🧩 **Your turn** — replace `None` with your code. A checker tells you whether it works. Hidden solutions follow.

### By the end you can
1. Implement a filter (convolution) from scratch and choose kernels for blurring, sharpening and edges.
2. Pick a threshold automatically with Otsu's method.
3. Estimate a background (from the frame border, or from many video frames) and subtract it.
4. Clean a mask with morphology and split it into objects with connected components.
5. Measure an object's area, position, orientation and length with principal component analysis (PCA).

This is the toolkit behind the **Mentee 2 segmentation baseline** (Tasks 5–6) and Module 01's pipeline.

In [ ]:
#@title ▶ Setup: scenes and real wildlife photographs (run it; you don't need to read it yet)
import numpy as np, matplotlib.pyplot as plt, urllib.request
from pathlib import Path
plt.rcParams.update({"figure.dpi": 100, "axes.titlesize": 10})

PHOTO_BASE = "https://raw.githubusercontent.com/ejri/aquatic-guardian-learning-modules/main/images/"
PHOTO_FILES = {
    "reef-school": "reef-school.jpg", "manatees": "manatees.jpg", "sea-lion": "sea-lion.jpg",
    "nurse-shark": "nurse-shark.jpg", "eagle-ray": "eagle-ray.jpg", "grouper": "grouper.jpg",
    "angelfish": "angelfish.jpg", "turtle": "turtle.jpg", "noaa-honu": "noaa-honu.jpg",
    "noaa-whale": "noaa-whale.jpg", "kelp-forest": "kelp-forest.jpg", "zebra-shark": "zebra-shark.jpg",
}

def load_photo(name, max_side=480):
    """Load a real wildlife JPEG. Looks in ./images and ../images, then downloads from GitHub."""
    file = PHOTO_FILES.get(name, name if str(name).endswith(".jpg") else str(name) + ".jpg")
    arr = None
    for candidate in (Path("images") / file, Path("../images") / file, Path(file)):
        if candidate.is_file():
            arr = plt.imread(candidate)
            break
    if arr is None:
        urllib.request.urlretrieve(PHOTO_BASE + file, file)
        arr = plt.imread(file)
    if getattr(arr, "dtype", None) == np.uint8:
        arr = arr.astype(float) / 255.0
    arr = np.asarray(arr[..., :3], dtype=float)
    h, w = arr.shape[:2]
    scale = max_side / max(h, w)
    if scale < 1:
        yy = (np.linspace(0, h - 1, max(1, int(h * scale)))).astype(int)
        xx = (np.linspace(0, w - 1, max(1, int(w * scale)))).astype(int)
        arr = arr[yy][:, xx]
    return arr


# Illustrative attenuation coefficients beta (1/m) for R, G, B, and the colour of the veiling light.
# These are round teaching numbers with the right ordering, not measurements of any real site.
WATER = {
    "clear_ocean":    dict(beta=(0.40, 0.07, 0.04), veil=(0.02, 0.20, 0.42)),
    "coastal":        dict(beta=(0.55, 0.20, 0.30), veil=(0.05, 0.32, 0.27)),
    "turbid_harbour": dict(beta=(0.95, 0.60, 0.75), veil=(0.14, 0.30, 0.22)),
}

def _fish_mask(H, W, cx, cy, L, ang, aspect=4.0):
    """Ellipse body plus a triangular tail. L is total length (snout to tail tip) in pixels."""
    yy, xx = np.mgrid[0:H, 0:W]
    a = L / 2.25; b = a / aspect
    X = (xx - cx) * np.cos(ang) + (yy - cy) * np.sin(ang)
    Y = -(xx - cx) * np.sin(ang) + (yy - cy) * np.cos(ang)
    body = (X / a) ** 2 + (Y / b) ** 2 <= 1
    tx = -X - 0.80 * a
    tail = (tx >= 0) & (tx <= 0.45 * a) & (np.abs(Y) <= 1.3 * b * tx / (0.45 * a))
    return body | tail, X, Y, a, b

FISH_COLOURS = [(0.97, 0.55, 0.12), (0.96, 0.84, 0.22), (0.80, 0.82, 0.84), (0.93, 0.35, 0.25)]

def make_scene(n_fish=4, H=120, W=180, seed=0, bg_seed=None, fish_dist=(2.0, 6.0), bg_dist=12.0,
               length_px=(22, 40), min_gap=0.55, touching=False, aspect=(4.0, 4.0)):
    """Returns a dict with the true scene radiance J (what you'd see in air), a distance map in metres,
    one boolean mask and one box (x1, y1, x2, y2) per fish, and per-fish truth."""
    r = np.random.default_rng(seed)
    rb = np.random.default_rng(seed if bg_seed is None else bg_seed)
    yy, xx = np.mgrid[0:H, 0:W] / max(H, W)
    J = np.stack([0.48 + 0.10 * yy, 0.43 + 0.08 * yy, 0.34 + 0.04 * xx], -1)       # reef wall / sand
    J = J + 0.04 * rb.standard_normal((H, W, 1)) + 0.015 * rb.standard_normal((H, W, 3))
    dist = np.full((H, W), float(bg_dist))
    masks, boxes, truth, placed = [], [], [], []
    for i in range(n_fish):
        for _ in range(200):
            L = r.uniform(*length_px); ang = r.uniform(-0.6, 0.6) + (np.pi if r.random() < 0.5 else 0)
            cx = r.uniform(0.15 * W, 0.85 * W); cy = r.uniform(0.18 * H, 0.82 * H)
            gap = 0.25 if touching else min_gap
            if all((cx - px) ** 2 + (cy - py) ** 2 > (gap * (L + pl)) ** 2 for px, py, pl in placed):
                break
        placed.append((cx, cy, L))
        asp = aspect[0] if aspect[0] == aspect[1] else r.uniform(*aspect)
        m, X, Y, a, b = _fish_mask(H, W, cx, cy, L, ang, asp)
        col = np.array(FISH_COLOURS[i % len(FISH_COLOURS)])
        J[m] = col
        stripe = m & (np.abs(X - 0.15 * a) < 0.09 * a)
        J[stripe] = (0.95, 0.95, 0.92)
        eye = (X - 0.62 * a) ** 2 + (Y - 0.15 * b) ** 2 < (0.12 * b + 0.6) ** 2
        J[eye & m] = (0.05, 0.05, 0.08)
        d = r.uniform(*fish_dist); dist[m] = d
        ys, xs = np.where(m)
        masks.append(m); boxes.append((int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1))
        truth.append(dict(cx=float(cx), cy=float(cy), length_px=float(L), angle=float(ang), dist_m=float(d), aspect=float(asp)))
    return dict(J=np.clip(J, 0, 1), dist=dist, masks=masks, boxes=boxes, truth=truth)

def underwater(J, dist, water="coastal", light=1.0, photons=None, seed=0):
    """Simple image-formation model, per colour channel c:
         I_c = light * ( J_c * t_c  +  B_c * (1 - t_c) ),   t_c = exp(-beta_c * distance)
    photons: if set, adds shot noise (fewer photons -> noisier). Lower `light` means fewer photons."""
    beta = np.array(WATER[water]["beta"]); B = np.array(WATER[water]["veil"])
    t = np.exp(-beta[None, None, :] * dist[..., None])
    I = light * (J * t + B * (1 - t))
    if photons:
        rng = np.random.default_rng(seed)
        n = photons * light
        I = rng.poisson(np.clip(I, 0, None) / max(light, 1e-6) * n) / n * light
    return np.clip(I, 0, 1)

def degrade(img, turbidity=0.0, low_light=0.0, blur_k=1):
    """The degradation utility used in the Aquatic Guardian project notebooks (copied verbatim in spirit)."""
    out = img.copy()
    if blur_k > 1:
        pad = blur_k // 2; P = np.pad(out, ((pad, pad), (pad, pad), (0, 0)), mode="edge"); acc = np.zeros_like(out)
        for dy in range(blur_k):
            for dx in range(blur_k): acc += P[dy:dy + out.shape[0], dx:dx + out.shape[1], :]
        out = acc / (blur_k * blur_k)
    if turbidity > 0:
        veil = np.array([0.20, 0.42, 0.45]); out = (1 - turbidity) * out + turbidity * veil
        out = (1 - 0.5 * turbidity) * out + 0.5 * turbidity * out.mean()
    if low_light > 0: out = (out * (1 - 0.85 * low_light)) ** (1 + 0.6 * low_light)
    return np.clip(out, 0, 1)

def show(images, titles=None, size=3.2, cmap=None):
    images = images if isinstance(images, (list, tuple)) else [images]
    fig, ax = plt.subplots(1, len(images), figsize=(size * len(images), size * 0.72))
    ax = np.atleast_1d(ax)
    for i, (a, im) in enumerate(zip(ax, images)):
        a.imshow(im, cmap=cmap if im.ndim == 2 else None, vmin=0 if im.ndim == 2 and cmap else None,
                 vmax=1 if im.ndim == 2 and cmap else None)
        a.set_title(titles[i] if titles else ""); a.axis("off")
    plt.tight_layout(); plt.show()

def draw_boxes(img, boxes, colour=(1.0, 0.85, 0.0)):
    out = img.copy()
    for (x1, y1, x2, y2) in boxes:
        x2 -= 1; y2 -= 1
        out[y1, x1:x2 + 1] = colour; out[y2, x1:x2 + 1] = colour
        out[y1:y2 + 1, x1] = colour; out[y1:y2 + 1, x2] = colour
    return out
print("Setup ready. Water types:", list(WATER))

In [ ]:
from scipy import ndimage
scene = make_scene(n_fish=5, seed=11)
img = underwater(scene["J"], scene["dist"], "coastal", photons=600, seed=0)
gray = img @ np.array([0.299, 0.587, 0.114])
show([img, gray], ["camera frame", "grayscale"], cmap="gray")

## 1. Filters: a small window slid across the image ▶

A filter replaces each pixel with a weighted sum of its neighbours. The weights form a **kernel**. Below is the
whole algorithm in a few lines: shift the padded image under each kernel position and add.

Strictly, this is *correlation*. True convolution flips the kernel first. For symmetric kernels they are
identical, and deep-learning libraries also compute correlation while calling it convolution.

In [ ]:
def conv2d(image, kernel):
    kh, kw = kernel.shape; ph, pw = kh // 2, kw // 2
    P = np.pad(image, ((ph, ph), (pw, pw)), mode="edge")
    out = np.zeros_like(image, dtype=float)
    for dy in range(kh):
        for dx in range(kw):
            out += kernel[dy, dx] * P[dy:dy + image.shape[0], dx:dx + image.shape[1]]
    return out

KERNELS = {
    "box blur": np.ones((5, 5)) / 25,
    "sharpen": np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], float),
    "Sobel x (vertical edges)": np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], float),
    "Sobel y (horizontal edges)": np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], float),
}
outs = {k: conv2d(gray, v) for k, v in KERNELS.items()}
show([np.clip(outs["box blur"], 0, 1), np.clip(outs["sharpen"], 0, 1),
      np.abs(outs["Sobel x (vertical edges)"]), np.abs(outs["Sobel y (horizontal edges)"])],
     list(KERNELS), size=2.9, cmap="gray")
same = np.allclose(conv2d(gray, KERNELS["sharpen"]), ndimage.correlate(gray, KERNELS["sharpen"], mode="nearest"))
print("matches scipy.ndimage.correlate:", same)

**Edges** are where brightness changes fast. Combine the two Sobel responses into a gradient magnitude
$\sqrt{G_x^2 + G_y^2}$. Fish outlines light up; so does noise, which is why we usually blur first.

In [ ]:
gx, gy = conv2d(gray, KERNELS["Sobel x (vertical edges)"]), conv2d(gray, KERNELS["Sobel y (horizontal edges)"])
smooth = conv2d(gray, np.ones((3, 3)) / 9)
gx2, gy2 = conv2d(smooth, KERNELS["Sobel x (vertical edges)"]), conv2d(smooth, KERNELS["Sobel y (horizontal edges)"])
show([np.hypot(gx, gy), np.hypot(gx2, gy2)], ["edges, raw", "edges after a 3x3 blur"], cmap="gray")

✏️ **Predict, then change.** Replace the 3×3 blur with a 9×9 blur. What happens to the noise, and to the
thin tail fins? There's always a trade-off between suppressing noise and keeping detail.

## 2. From colour to a single "unusual-ness" map ▶

Fish are easier to separate by colour difference from the background than by brightness alone. As in the
Mentee notebooks, we estimate the background colour from the frame border.

In [ ]:
border = np.concatenate([img[:6].reshape(-1, 3), img[-6:].reshape(-1, 3), img[:, :6].reshape(-1, 3), img[:, -6:].reshape(-1, 3)])
bg_colour = np.median(border, axis=0)
diff = np.sqrt(((img - bg_colour) ** 2).sum(-1))
plt.figure(figsize=(5, 2.6)); plt.hist(diff.ravel(), bins=80); plt.yscale("log")
plt.xlabel("difference from background colour"); plt.ylabel("pixels (log)"); plt.title("Two populations: background and fish"); plt.show()

## 3. Choosing the threshold automatically: Otsu's method ▶

Otsu tries every threshold and keeps the one that best separates the histogram into two groups (it maximises the
*between-class variance* $w_0 w_1 (\mu_0 - \mu_1)^2$). Here is the full method.

In [ ]:
def otsu(values, bins=256):
    lo, hi = values.min(), values.max()
    hist, edges = np.histogram(values, bins=bins, range=(lo, hi))
    p = hist / hist.sum(); centres = (edges[:-1] + edges[1:]) / 2
    w0 = np.cumsum(p); mu = np.cumsum(p * centres); muT = mu[-1]
    between = (muT * w0 - mu) ** 2 / (w0 * (1 - w0) + 1e-12)
    return centres[np.argmax(between)]

from skimage.filters import threshold_otsu
thr = otsu(diff)
print(f"our Otsu threshold: {thr:.3f}   scikit-image: {threshold_otsu(diff):.3f}")
mask = diff > thr
show([diff, mask], ["difference map", f"mask (diff > {thr:.2f})"], cmap="gray")

## 4. Cleaning the mask: morphology ▶

- **Opening** (erode, then dilate) deletes specks smaller than the structuring element.
- **Closing** (dilate, then erode) fills small holes, like the white stripe on our fish.

In [ ]:
opened = ndimage.binary_opening(mask, structure=np.ones((3, 3)))
cleaned = ndimage.binary_closing(opened, structure=np.ones((3, 3)))
show([mask, opened, cleaned], ["raw mask", "after opening", "after opening + closing"], cmap="gray")

## 5. Connected components: one label per object ▶

Pixels that touch belong to the same object. Below is a plain flood-fill, the same algorithm as the Mentee 2
notebook's `label_cc`, followed by SciPy's version, which runs in compiled code.

In [ ]:
def label_cc(m):
    H, W = m.shape; lab = np.zeros((H, W), int); cur = 0
    for i in range(H):
        for j in range(W):
            if m[i, j] and lab[i, j] == 0:
                cur += 1; stack = [(i, j)]; lab[i, j] = cur
                while stack:
                    y, x = stack.pop()
                    for dy, dx in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                        ny, nx = y + dy, x + dx
                        if 0 <= ny < H and 0 <= nx < W and m[ny, nx] and lab[ny, nx] == 0:
                            lab[ny, nx] = cur; stack.append((ny, nx))
    return lab, cur

import time
t0 = time.time(); lab_py, n_py = label_cc(cleaned); t_py = time.time() - t0
t0 = time.time(); lab_sp, n_sp = ndimage.label(cleaned); t_sp = time.time() - t0
print(f"flood fill: {n_py} objects in {t_py*1000:.0f} ms   scipy: {n_sp} objects in {t_sp*1000:.1f} ms")
colours = np.random.default_rng(1).random((n_sp + 1, 3)); colours[0] = 0
show([img, colours[lab_sp]], ["frame", f"{n_sp} labelled components"])

## 6. Measuring shape with PCA ▶

Treat each object's pixel coordinates as a cloud of points. The covariance of that cloud has two eigenvectors:
the **major axis** (direction of greatest spread, along the fish) and the **minor axis** (across it). Projecting
pixels onto the major axis and taking the extent gives the length in pixels. This is exactly `measure()` in the
Mentee 2 notebook.

In [ ]:
def measure(lab, k):
    ys, xs = np.where(lab == k)
    if len(xs) < 20: return None
    X = np.stack([xs - xs.mean(), ys - ys.mean()])
    evals, evecs = np.linalg.eigh(np.cov(X))
    major, minor = evecs[:, np.argmax(evals)], evecs[:, np.argmin(evals)]
    along, across = X.T @ major, X.T @ minor
    L, Wd = along.max() - along.min(), across.max() - across.min()
    return dict(area=len(xs), cx=xs.mean(), cy=ys.mean(), length_px=L, width_px=Wd, aspect=L / max(Wd, 1e-6),
                angle_deg=np.degrees(np.arctan2(major[1], major[0])))

import pandas as pd
rows = []
for k in range(1, n_sp + 1):
    m = measure(lab_sp, k)
    if m is None: continue
    j = int(np.argmin([(t["cx"] - m["cx"]) ** 2 + (t["cy"] - m["cy"]) ** 2 for t in scene["truth"]]))
    m["true_length_px"] = scene["truth"][j]["length_px"]; rows.append(m)
table = pd.DataFrame(rows).round(1)
table["length_error_%"] = (100 * (table.length_px - table.true_length_px) / table.true_length_px).round(1)
table[["cx", "cy", "area", "length_px", "true_length_px", "length_error_%", "aspect", "angle_deg"]]

✏️ **Interpret:** is the length error mostly positive or negative? Think about what happens to the thin tail
and to edge pixels whose colour is blended with the water. A consistent bias here becomes a consistent bias in
biomass (Module 07 shows it gets roughly **tripled**).

## 7. The known failure: touching fish ▶

Connected components can't separate fish that touch. One fix: the **distance transform** (each foreground pixel's
distance to the nearest background pixel) has one peak per body; use those peaks as seeds for a **watershed**.

In [ ]:
from skimage.segmentation import watershed
from skimage.feature import peak_local_max
crowd = make_scene(n_fish=6, seed=8, touching=True)
cimg = underwater(crowd["J"], crowd["dist"], "coastal")
cdiff = np.sqrt(((cimg - np.median(cimg[:5].reshape(-1, 3), 0)) ** 2).sum(-1))
cmask = ndimage.binary_closing(ndimage.binary_opening(cdiff > otsu(cdiff)), np.ones((3, 3)))
lab_cc, n_cc = ndimage.label(cmask)
dist_map = ndimage.gaussian_filter(ndimage.distance_transform_edt(cmask), 2)   # smoothed: one ridge per body
peaks = peak_local_max(dist_map, min_distance=10, labels=lab_cc, exclude_border=False)
markers = np.zeros_like(lab_cc); markers[tuple(peaks.T)] = np.arange(1, len(peaks) + 1)
lab_ws = watershed(-dist_map, markers, mask=cmask)
pal = np.random.default_rng(3).random((max(n_cc, lab_ws.max()) + 1, 3)); pal[0] = 0
def count_big(lab, min_area=30):
    return int(sum((lab == k).sum() >= min_area for k in range(1, lab.max() + 1)))
show([cimg, pal[lab_cc], pal[lab_ws]], ["6 fish, some touching",
     f"connected components: {count_big(lab_cc)} objects", f"watershed: {count_big(lab_ws)} objects"])
print("Counts include only objects of at least 30 pixels, so specks are ignored.")
print("Look closely at the frame: is every fish in the mask at all? What happened to the darkest one?")

✏️ Change `min_distance` to 4 and re-run. Watershed often helps, but with loose settings it over-splits a single fish. This is the honest limit of classical
segmentation and the reason the Mentee 2 notebook's Task 10 upgrades to learned instance segmentation.

## 8. Video: a better background from many frames ▶

With a fixed camera, the **median of many frames** at each pixel is the background, because each fish occupies a
given pixel only briefly. This is how many fish-counting systems start.

In [ ]:
frames = [underwater(make_scene(n_fish=4, seed=100 + i, bg_seed=7)["J"], np.full((120, 180), 4.0), "coastal") for i in range(15)]
median_bg = np.median(np.stack(frames), axis=0)
fg = np.sqrt(((frames[0] - median_bg) ** 2).sum(-1))
show([frames[0], median_bg, fg], ["frame 0 of 15", "per-pixel median background", "frame 0 minus background"], cmap="gray")

### 🧩 Exercise — a fish-shape filter

Fish are elongated; crabs and rocks are not. Using `table` from section 6, write a boolean Series
`elongated` that is `True` where `aspect` is at least 2.5. Then count how many objects pass.

In [ ]:
elongated = None   # YOUR CODE, e.g. table["aspect"] >= ...

def check_elong(e):
    if e is None: print("Not yet: replace None."); return
    exp = table["aspect"] >= 2.5
    if hasattr(e, "equals") and e.equals(exp): print(f"✅ Correct: {int(exp.sum())} of {len(table)} objects pass.")
    else: print("❌ Not quite. Compare with table['aspect'] >= 2.5.")
check_elong(elongated)

In [ ]:
#@title Solution (double-click to show)
elongated = table["aspect"] >= 2.5
check_elong(elongated)

## Check yourself

<details><summary>1. Why blur before computing edges?</summary>Edge kernels amplify pixel-to-pixel noise; blurring suppresses noise first, at the cost of some fine detail.</details>
<details><summary>2. What does Otsu assume about the histogram?</summary>That it's roughly two groups (background and objects). It struggles when objects are tiny, or when there are more than two populations.</details>
<details><summary>3. Two fish touch and connected components returns one object with double the area. Name two fixes.</summary>Distance transform + watershed, or a learned instance-segmentation model. Better camera siting (fewer overlaps) helps too.</details>
<details><summary>4. Why is a median background better than a mean for video?</summary>The median ignores the occasional frames where a fish covers the pixel; the mean gets pulled toward fish colours.</details>

**Next:** Module 04, *Learning to classify* — from hand-made rules to a model that learns its own decision
boundary, and how to evaluate it without fooling yourself.